# Pseudopotentials and Nonlocal Projectors

Real pseudopotential files separate core electrons from valence electrons.  In
the current implementation:

$$
V_\mathrm{ps}=V_\mathrm{local}+\sum_{ij}|\beta_i\rangle D_{ij}\langle\beta_j|.
$$

The local part is applied on the real-space grid.  The separable nonlocal part
is represented and applied as a correctness-first projector operator.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import mlx.core as mx
import numpy as np


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("could not locate repository root")


ROOT = find_repo_root()


## Why pseudopotentials exist

An all-electron Hamiltonian must resolve rapidly oscillating core states near
the nucleus.  A pseudopotential replaces the nucleus plus frozen core electrons
with an effective ion potential seen by the valence electrons:

$$
V_\mathrm{ion}
\;\longrightarrow\;
V_\mathrm{ps}
= V_\mathrm{local}(r) + V_\mathrm{nonlocal}.
$$

This makes a plane-wave/grid calculation far cheaper because the valence
wavefunctions are smoother near the ion centers.  The tradeoff is that the
pseudopotential format and projector conventions become part of the numerical
contract.


## Parse UPF and GTH reference inputs

The vendored QE/CP2K trees are reference data only.  We parse the files directly
without importing or linking those packages.


### Local radial potentials

UPF files usually provide tabulated radial data:

$$
\{r_m, V_\mathrm{local}(r_m)\}_{m=1}^{M}.
$$

The code interpolates that radial function onto the periodic real-space grid by
using the minimum-image ion displacement:

$$
r_I(g)=|r_g-R_I|_\mathrm{MIC},
\qquad
V_\mathrm{local}(r_g)
= \sum_I V_I^\mathrm{local}(r_I(g)).
$$

GTH local potentials are compact analytic functions with Gaussian damping, so
they can be evaluated directly from the parsed coefficients.


In [ ]:
from mlx_atomistic.dft import (
    DFTSystem,
    Ion,
    IonCollection,
    NonlocalPseudopotentialOperator,
    SCFConfig,
    read_gth,
    read_upf,
    run_scf,
)

upf = read_upf(ROOT / "vendors/quantum-espresso/pseudo/Si_r.upf")
gth = read_gth(ROOT / "vendors/quantum-espresso/pseudo/H-q1.gth", element="H")

metadata = [
    {
        "format": str(item.format),
        "element": item.element,
        "valence": item.valence_charge,
        "projectors": len(item.nonlocal_projectors),
        "nonlocal_available": item.nonlocal_available,
    }
    for item in (upf, gth)
]
metadata


In [ ]:
r = np.linspace(1e-4, 4.0, 400)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(r, upf.local_potential(r), label="Si UPF local")
ax.plot(r, gth.local_potential(r), label="H GTH local")
ax.axhline(0.0, color="black", linewidth=1)
ax.set_title("radial local pseudopotentials")
ax.set_xlabel(r"$r$ / bohr")
ax.set_ylabel(r"$V_\mathrm{local}(r)$ / Ha")
ax.legend()
fig.tight_layout()


## Inspect projector action

The nonlocal operator is Hermitian by construction:

$$
V_\mathrm{NL}\psi = \sum_i |\beta_i\rangle D_i \langle\beta_i|\psi\rangle.
$$

The plot below shows the normalized real-space projector fields generated from
the parsed UPF metadata.


### Separable nonlocal energy

For occupied orbitals \(\psi_n\) with occupations \(f_n\), the diagonal
separable form used here contributes

$$
E_\mathrm{NL}
= \sum_n f_n
\sum_i D_i\left|\langle \beta_i|\psi_n\rangle\right|^2.
$$

The corresponding operator action is

$$
V_\mathrm{NL}\psi_n
= \sum_i |\beta_i\rangle D_i\langle\beta_i|\psi_n\rangle.
$$

The important implementation checks are: the projector fields are normalized on
the grid, the operator is Hermitian, and the energy appears exactly once in the
SCF decomposition.


In [ ]:
ions = IonCollection([Ion("Si", (4.0, 4.0, 4.0), upf)])
system = DFTSystem(cell=(8.0, 8.0, 8.0), grid_shape=(4, 4, 4), ions=ions)
operator = NonlocalPseudopotentialOperator.from_ions(ions, system.grid)
print(operator.to_dict())

projectors = np.asarray(operator.projectors.projectors)
fig, axes = plt.subplots(1, min(operator.projectors.count, 4), figsize=(12, 3))
if operator.projectors.count == 1:
    axes = [axes]
z = projectors.shape[-1] // 2
for index, ax in enumerate(axes):
    im = ax.imshow(projectors[index, :, :, z].T, origin="lower", cmap="viridis")
    ax.set_title(f"projector {index}")
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout()


## Local-only versus local + nonlocal SCF

This is a small-grid diagnostic, not a production calculation.  The useful
question is whether the nonlocal term appears exactly once in the energy
decomposition and changes the Hamiltonian consistently.


### Force provenance

Local ion forces can be evaluated from the local potential derivative:

$$
F_I^\mathrm{local}
= -\frac{\partial}{\partial R_I}
\int \rho(r)V_I^\mathrm{local}(|r-R_I|)\,dr.
$$

The nonlocal force path in this milestone is intentionally conservative: it
uses finite differences of the nonlocal energy rather than claiming a fully
optimized analytic projector-force implementation.  Diagnostics therefore
record force provenance as local analytic, nonlocal finite-difference, and
center-center contributions.


In [ ]:
base_config = SCFConfig(max_iterations=2, solver="dense", seed=7, convergence_mode="either")
local_only = run_scf(
    system,
    config=SCFConfig(**(base_config.__dict__ | {"apply_nonlocal": False})),
)
with_nonlocal = run_scf(
    system,
    config=SCFConfig(**(base_config.__dict__ | {"apply_nonlocal": True})),
)

labels = ["local only", "local + nonlocal"]
energies = [local_only.total_energy, with_nonlocal.total_energy]
nonlocal_terms = [
    local_only.energy_by_term.get("nonlocal_pseudopotential", 0.0),
    with_nonlocal.energy_by_term.get("nonlocal_pseudopotential", 0.0),
]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(labels, energies)
axes[0].set_title("total energy")
axes[0].set_ylabel("Ha")
axes[1].bar(labels, nonlocal_terms, color="tab:orange")
axes[1].set_title("nonlocal energy term")
axes[1].set_ylabel("Ha")
fig.tight_layout()

print(
    "local-only diagnostics:",
    local_only.to_dict()["nonlocal_applied"],
    local_only.energy_by_term,
)
print(
    "with-nonlocal diagnostics:",
    with_nonlocal.to_dict()["nonlocal_applied"],
    with_nonlocal.energy_by_term,
)
